# Time travel × external mappers — case studies (human; marketing appendix)

This notebook turns the aggregate curves/heatmaps into **explainable examples** that work well in a Results narrative.

## What it does

- Finds identifiers where a chosen external tool fails **without time travel** (returns `1:0`) but IDTrack succeeds.
- Exports a manuscript-ready table of representative examples.
- Optionally (small N), extracts IDTrack `explain=True` audit summaries to show *why* these are legitimate time-travel mappings.

## Why it markets IDTrack

A reviewer will accept the quantitative figure more readily if you can point to a handful of concrete, auditable failures that align with the conceptual framing:

- External mappers are point-in-time services (good at “now”).
- Historical identifiers are a *different problem*; you need an explicit time coordinate system and a snapshot boundary.

This notebook is cache-first and expects the stage-0 cache notebook to have been executed.


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

import sys

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / 'idtrack').is_dir() and (REPO_ROOT / 'idtrack-manuscript').is_dir()):
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENTS_SRC = REPO_ROOT / 'idtrack' / 'reproducibility' / 'experiments' / 'src'
sys.path.append(str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    atomic_write_dataframe_csv,
    atomic_write_dataframe_latex,
    notebook_context,
    read_json,
    read_pickle,
)

from idtrack_results import (  # noqa: E402
    external_df_to_output_sets,
    matchings_to_output_sets,
)

ctx = notebook_context('time_travel_vs_external_mappers', start=REPO_ROOT)
CACHE_DIR = ctx.experiment_cache
MANUSCRIPT_TABLES = ctx.manuscript_tables
EXPERIMENT_TABLES = (ctx.experiment_outputs / 'tables')

print('CACHE_DIR:', CACHE_DIR)


In [ ]:
# -------------------- Load the latest cache fingerprint --------------------

params_candidates = list(CACHE_DIR.glob('time_travel_vs_external_mappers_params_*.json'))
if not params_candidates:
    raise FileNotFoundError(f'No params JSON found under {CACHE_DIR}. Run the stage-0 cache notebook.')

PARAMS_JSON = max(params_candidates, key=lambda p: p.stat().st_mtime)
fp = PARAMS_JSON.stem.split('_')[-1]
params = read_json(PARAMS_JSON)

bundles = []
for p in sorted(CACHE_DIR.glob(f'time_travel_vs_external_mappers_bundle_{fp}_from*_b*.pickle')):
    bundles.append(read_pickle(p))

print('Fingerprint:', fp)
print('Loaded bundles:', len(bundles))
print('Targets:', params.get('target_databases'))
print('External methods:', params.get('external_methods'))


In [ ]:
# -------------------- Choose the case-study lens --------------------

TARGET_DB = 'HGNC Symbol'
METHOD = 'pybiomart'   # compare: 'mygene', 'gprofiler', 'gget'

# How many case-study rows to export
N_EXAMPLES = 30

print('TARGET_DB:', TARGET_DB)
print('METHOD:', METHOD)


In [ ]:
# -------------------- Collect failure examples --------------------

rows = []
for bundle in bundles:
    fr = int(bundle.get('from_release'))
    b = int(bundle.get('bootstrap'))
    ids_from = [str(x) for x in (bundle.get('ids_from') or [])]

    idt_old = (bundle.get('idtrack_old_to_target_matchings') or {}).get(TARGET_DB) or []
    ref_sets = matchings_to_output_sets(idt_old)

    ext_naive = ((bundle.get('external_results') or {}).get('naive') or {}).get(TARGET_DB) or {}
    df = ext_naive.get(METHOD)
    if df is None:
        continue
    ext_sets = external_df_to_output_sets(df, inputs=ids_from)

    # Backbone to_release IDs (for narrative; optional)
    backbone = bundle.get('backbone_matchings') or []
    to_map = {}
    for item in backbone:
        q = str(item.get('query_id'))
        if not q or q.lower() in {'nan', 'none', 'null'}:
            continue
        if item.get('no_corresponding') or item.get('no_conversion'):
            continue
        targets = item.get('target_id') or []
        uniq = []
        seen = set()
        for t in targets:
            if t is None:
                continue
            s = str(t).strip()
            if not s or s.lower() in {'nan', 'none', 'null'}:
                continue
            if s not in seen:
                seen.add(s)
                uniq.append(s)
        if len(uniq) == 1:
            to_map[q] = uniq[0]

    for q in ids_from:
        idt = ref_sets.get(q, set())
        ext = ext_sets.get(q, set())
        if (not ext) and idt:
            rows.append(
                {
                    'from_release': fr,
                    'bootstrap': b,
                    'query_ensembl_gene': q,
                    'to_release_ensembl_gene_1to1': to_map.get(q),
                    'idtrack_targets': ';'.join(sorted(idt))[:4000],
                    f'{METHOD}_targets': ';'.join(sorted(ext))[:4000],
                    'n_idtrack_targets': len(idt),
                    f'n_{METHOD}_targets': len(ext),
                }
            )

df_cases = pd.DataFrame(rows)
print('Candidate rows:', len(df_cases))
df_cases.head()


In [ ]:
# -------------------- Select representative examples --------------------

if df_cases.empty:
    print('No case studies found (check optional dependency availability, or change METHOD/TARGET_DB).')
else:
    # Prefer older releases first (more striking), then keep one bootstrap's worth.
    df_cases = df_cases.sort_values(['from_release', 'bootstrap']).reset_index(drop=True)
    df_out = df_cases.head(int(N_EXAMPLES)).copy()

    out_csv = MANUSCRIPT_TABLES / f'time_travel_vs_external_mappers_case_studies_{METHOD}_hgnc.csv'
    atomic_write_dataframe_csv(df_out, out_csv, index=False)
    atomic_write_dataframe_csv(df_out, EXPERIMENT_TABLES / out_csv.name, index=False)

    # LaTeX: keep it compact for manuscript inclusion (you may still hand-edit columns).
    out_tex = MANUSCRIPT_TABLES / f'time_travel_vs_external_mappers_case_studies_{METHOD}_hgnc.tex'
    atomic_write_dataframe_latex(
        df_out[
            [
                'from_release',
                'query_ensembl_gene',
                'to_release_ensembl_gene_1to1',
                'idtrack_targets',
                f'{METHOD}_targets',
            ]
        ].fillna(''),
        out_tex,
        index=False,
        escape=True,
        longtable=False,
    )
    atomic_write_dataframe_latex(
        df_out[
            [
                'from_release',
                'query_ensembl_gene',
                'to_release_ensembl_gene_1to1',
                'idtrack_targets',
                f'{METHOD}_targets',
            ]
        ].fillna(''),
        EXPERIMENT_TABLES / out_tex.name,
        index=False,
        escape=True,
        longtable=False,
    )

    print('Wrote:', out_csv)
    print('Wrote:', out_tex)

df_out.head(10) if 'df_out' in globals() else None


In [ ]:
# -------------------- Optional: extract explain=True audit summaries (small N) --------------------

# This section is intentionally small to keep it manageable.
# It builds the graph (memory-heavy) and requests explainability for a few examples.

N_EXPLAIN = 5

if 'df_out' not in globals() or df_cases.empty:
    print('No selected examples; skipping explainability.')
else:
    import idtrack

    IDTRACK_LOCAL_REPO = ctx.idtrack_local_repo
    api = idtrack.API(local_repository=str(IDTRACK_LOCAL_REPO))
    api.configure_logger()

    organism, latest = api.resolve_organism('human')
    snapshot = int(params.get('snapshot_release', 114))
    to_release = int(params.get('to_release', 114))
    if to_release > snapshot:
        raise ValueError(f'to_release={to_release} exceeds snapshot_release={snapshot}')

    api.build_graph(organism_name=organism, snapshot_release=snapshot, calculate_caches=True)

    explain_rows = []
    for _, row in df_out.head(int(N_EXPLAIN)).iterrows():
        q = str(row['query_ensembl_gene'])
        fr = int(row['from_release'])
        try:
            res = api.convert_identifier(
                q,
                from_release=fr,
                to_release=to_release,
                final_database=TARGET_DB,
                strategy=str(params.get('strategy', 'all')),
                explain=True,
                verbose=False,
            )
        except Exception as e:  # noqa: BLE001
            explain_rows.append({'query_ensembl_gene': q, 'from_release': fr, 'error': str(e)})
            continue

        explain_payload = res.get('explain')
        explain_rows.append(
            {
                'query_ensembl_gene': q,
                'from_release': fr,
                'to_release': to_release,
                'final_database': TARGET_DB,
                'target_id': ';'.join([str(x) for x in (res.get('target_id') or [])])[:4000],
                'no_target': bool(res.get('no_target', False)),
                'no_corresponding': bool(res.get('no_corresponding', False)),
                'no_conversion': bool(res.get('no_conversion', False)),
                'strategy': res.get('strategy'),
                'path_len': len(explain_payload or []) if isinstance(explain_payload, list) else None,
            }
        )

    df_explain = pd.DataFrame(explain_rows)
    df_explain


# Marketing extension: prevalence of external 1:0 where IDTrack succeeds

Case studies are persuasive, but a Results section often benefits from a simple prevalence statement.

This section quantifies (as a function of `from_release`) how often an external backend returns **1:0**
while IDTrack returns a **non-empty** target set for the same query IDs.

This is a clean marketing message because it does not claim biological “accuracy” — it quantifies *recoverability*.


In [ ]:
import matplotlib.pyplot as plt

from experiments_utils import EXTERNAL_MAPPER_METHODS_ORDERED, MANUSCRIPT_COLORS, label_panels, save_figure  # noqa: E402
from plotting_utils import heatmap  # noqa: E402

targets = list(params.get('target_databases') or [])
methods = [str(m).strip().lower() for m in (params.get('external_methods') or [])]

rows = []
for bundle in bundles:
    fr = int(bundle.get('from_release'))
    b = int(bundle.get('bootstrap'))
    ids_from = [str(x) for x in (bundle.get('ids_from') or [])]
    if not ids_from:
        continue

    for target_db in targets:
        idt_old = (bundle.get('idtrack_old_to_target_matchings') or {}).get(target_db) or []
        ref_sets = matchings_to_output_sets(idt_old)

        ext_naive = ((bundle.get('external_results') or {}).get('naive') or {}).get(target_db) or {}
        for method in methods:
            df = ext_naive.get(method)
            ext_sets = external_df_to_output_sets(df, inputs=ids_from)

            n_total = len(ids_from)
            n_ext0_idt1 = 0
            for q in ids_from:
                idt_ok = bool(ref_sets.get(q, set()))
                ext_ok = bool(ext_sets.get(q, set()))
                if (not ext_ok) and idt_ok:
                    n_ext0_idt1 += 1

            rows.append(
                {
                    'from_release': fr,
                    'bootstrap': b,
                    'target_db': target_db,
                    'method': method,
                    'n_total': n_total,
                    'n_ext0_idt1': n_ext0_idt1,
                    'frac_ext0_idt1': n_ext0_idt1 / n_total if n_total else float('nan'),
                }
            )

failure = pd.DataFrame(rows)
if failure.empty:
    print('No failure rows computed; check bundles / optional dependency availability.')
else:
    agg_fail = (
        failure.groupby(['target_db', 'method', 'from_release'], as_index=False)
        .agg(mean_frac_ext0_idt1=('frac_ext0_idt1', 'mean'), std_frac_ext0_idt1=('frac_ext0_idt1', 'std'))
    )

    out_csv = MANUSCRIPT_TABLES / 'time_travel_vs_external_mappers_external_failures_where_idtrack_succeeds.csv'
    atomic_write_dataframe_csv(agg_fail, out_csv, index=False)
    atomic_write_dataframe_csv(agg_fail, EXPERIMENT_TABLES / out_csv.name, index=False)
    print('Wrote:', out_csv)

    # Heatmap figure (one panel per target)
    targets_plot = targets[:2] if targets else sorted(agg_fail['target_db'].unique().tolist())
    ncols = max(1, len(targets_plot))
    fig, axes = plt.subplots(
        1,
        ncols,
        figsize=(6.8 * ncols, 3.2 + 0.35 * len(set(agg_fail['method']))),
        constrained_layout=True,
    )
    if ncols == 1:
        axes = [axes]

    for ax, target_db in zip(axes, targets_plot):
        sub = agg_fail[agg_fail['target_db'] == target_db].copy()
        if sub.empty:
            ax.axis('off')
            ax.text(0.5, 0.5, f'No data for {target_db}', ha='center', va='center')
            continue
        ordered = [m.lower() for m in EXTERNAL_MAPPER_METHODS_ORDERED if m.lower() in set(sub['method'])]
        ordered += [m for m in sorted(set(sub['method'])) if m not in set(ordered)]
        mat = sub.pivot(index='method', columns='from_release', values='mean_frac_ext0_idt1').reindex(ordered)
        vmax = float(np.nanmax(mat.values)) if mat.size else 1.0
        heatmap(
            ax,
            mat,
            title=f'External 1:0 where IDTrack succeeds\n{target_db}',
            cmap='Reds',
            vmin=0.0,
            vmax=vmax,
            square=False,
            cbar=True,
            cbar_label='fraction',
        )
        ax.set_xlabel('from_release')
        ax.set_ylabel('method')

    label_panels(axes)
    written = save_figure(
        fig,
        'fig_time_travel_vs_external_mappers_external_failures_where_idtrack_succeeds.pdf',
        ctx,
        formats=('pdf',),
    )
    print('Saved:', written['pdf'])
